<a href="https://www.kaggle.com/code/adegbaju/agentic-ai-brain-tumor-mri-classification?scriptVersionId=310998970" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

 
# BRAIN TUMOR MRI CLASSIFICATION WITH AGENTIC AI
 ============================================

Complete Kaggle Notebook Implementation

Brain Tumor MRI Classification with Agentic AI

Autonomous Deep Learning System with Multi-Agent Architecture

## Environment Setup and Installation

In [1]:
# Install required packages
!pip install -q timm albumentations optuna plotly kaleido torchmetrics pytorch-lightning grad-cam
!pip install -q transformers datasets accelerate
!pip install -q torchvision torchaudio --upgrade
!pip install torch-lr-finder

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 54.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 114.2 MB/s eta 0

In [2]:
# Import necessary libraries
import os
import gc
import re
import sys
import json
import time
import math
import random
import shutil
import warnings
import datetime
from pathlib import Path
from collections import defaultdict, Counter
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Any, Union, Callable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps, ImageFilter, ImageEnhance
import cv2

# Deep Learning imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW, Adam, SGD
from torch.optim.lr_scheduler import (
    CosineAnnealingLR, ReduceLROnPlateau, OneCycleLR,
    CosineAnnealingWarmRestarts, StepLR, ExponentialLR
)
from torch.cuda.amp import autocast, GradScaler

# Computer Vision and Model imports
import torchvision
import torchvision.transforms as T
from torchvision import models
from torchvision.models import (
    resnet50, resnet101, resnet152,
    densenet121, densenet169, densenet201,
    efficientnet_b0, efficientnet_b3, efficientnet_b5, efficientnet_b7,
    vit_b_16, vit_l_16, swin_t, swin_s, swin_b,
    convnext_tiny, convnext_base, convnext_large,
    maxvit_t, inception_v3, ResNet50_Weights
)

# Transformers
from transformers import (
    AutoImageProcessor, AutoModelForImageClassification,
    DeiTForImageClassification, SwinForImageClassification,
    ConvNextForImageClassification, ViTForImageClassification
)

# Metrics and Evaluation
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report, roc_auc_score,
    cohen_kappa_score, matthews_corrcoef
)
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.preprocessing import label_binarize
import torchmetrics

# Visualization and Plotting
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio

# Image Augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Progress bars and utilities
from tqdm.auto import tqdm
import optuna
from torch_lr_finder import LRFinder

# For model interpretation
from pytorch_grad_cam import GradCAM, ScoreCAM, GradCAMPlusPlus, AblationCAM, XGradCAM, EigenCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Check GPU availability
print("=" * 80)
print("SYSTEM CONFIGURATION")
print("=" * 80)
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    !nvidia-smi --query-gpu=memory.total,memory.free,memory.used --format=csv
print("=" * 80)


SYSTEM CONFIGURATION
Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch version: 2.11.0+cu130
CUDA available: True
CUDA version: 13.0
GPU device: Tesla T4
GPU memory: 15.64 GB
memory.total [MiB], memory.free [MiB], memory.used [MiB]
15360 MiB, 14910 MiB, 3 MiB
15360 MiB, 14910 MiB, 3 MiB


##  Agentic AI Framework - Intelligent Autonomous Agents

In [3]:
@dataclass
class AgentConfig:
    """Configuration for AI agents"""
    name: str
    role: str
    capabilities: List[str]
    confidence_threshold: float = 0.85
    max_retries: int = 3
    learning_rate: float = 0.01

class BaseAgent:
    """Base class for all intelligent agents"""
    
    def __init__(self, config: AgentConfig):
        self.config = config
        self.memory = []
        self.knowledge_base = {}
        self.performance_history = []
        
    def perceive(self, data: Any) -> Dict:
        """Perceive and analyze input data"""
        raise NotImplementedError
    
    def reason(self, perception: Dict) -> Dict:
        """Reason about perceived data and make decisions"""
        raise NotImplementedError
    
    def act(self, decision: Dict) -> Any:
        """Execute actions based on reasoning"""
        raise NotImplementedError
    
    def learn(self, feedback: Dict):
        """Learn from feedback and update knowledge"""
        self.memory.append(feedback)
        self.performance_history.append(feedback.get('performance', 0))
        
    def communicate(self, message: Dict) -> Dict:
        """Communicate with other agents"""
        return {
            'source': self.config.name,
            'timestamp': datetime.datetime.now().isoformat(),
            'message': message,
            'confidence': self.config.confidence_threshold
        }

class DataAnalysisAgent(BaseAgent):
    """Agent responsible for data exploration and analysis"""
    
    def __init__(self, config: AgentConfig):
        super().__init__(config)
        self.data_stats = {}
        self.anomalies = []
        self.recommendations = []
        
    def perceive(self, data_path: str) -> Dict:
        """Analyze dataset structure and characteristics"""
        perception = {
            'dataset_path': data_path,
            'classes': [],
            'image_counts': {},
            'image_sizes': [],
            'color_distributions': [],
            'potential_issues': []
        }
        
        # Analyze training data
        train_path = Path(data_path) / 'Training'
        test_path = Path(data_path) / 'Testing'
        
        for class_name in os.listdir(train_path):
            class_path = train_path / class_name
            if class_path.is_dir():
                images = list(class_path.glob('*.jpg')) + list(class_path.glob('*.png'))
                perception['classes'].append(class_name)
                perception['image_counts'][f'train_{class_name}'] = len(images)
                
                # Sample images for size analysis
                sizes = []
                for img_path in images[:100]:  # Sample 100 images
                    try:
                        with Image.open(img_path) as img:
                            sizes.append(img.size)
                            # Check color mode
                            if img.mode not in ['RGB', 'L']:
                                perception['potential_issues'].append(
                                    f"Unusual color mode {img.mode} in {img_path}"
                                )
                    except Exception as e:
                        perception['potential_issues'].append(f"Corrupted image: {img_path}")
                
                perception['image_sizes'].extend(sizes)
        
        # Analyze test data
        for class_name in os.listdir(test_path):
            class_path = test_path / class_name
            if class_path.is_dir():
                images = list(class_path.glob('*.jpg')) + list(class_path.glob('*.png'))
                perception['image_counts'][f'test_{class_name}'] = len(images)
        
        self.data_stats = perception
        return perception
    
    def reason(self, perception: Dict) -> Dict:
        """Reason about data quality and preprocessing needs"""
        decision = {
            'data_quality_score': 0.0,
            'recommended_preprocessing': [],
            'augmentation_strategy': [],
            'potential_biases': [],
            'class_balance_status': 'balanced'
        }
        
        # Check class balance
        train_counts = [v for k, v in perception['image_counts'].items() if k.startswith('train')]
        if train_counts:
            max_count = max(train_counts)
            min_count = min(train_counts)
            if max_count - min_count > min_count * 0.1:
                decision['class_balance_status'] = 'imbalanced'
                decision['recommended_preprocessing'].append('class_weighting')
        
        # Analyze image sizes
        sizes = perception['image_sizes']
        if sizes:
            width_std = np.std([s[0] for s in sizes])
            height_std = np.std([s[1] for s in sizes])
            if width_std > 100 or height_std > 100:
                decision['recommended_preprocessing'].append('resize_to_uniform')
                decision['data_quality_score'] += 0.7
            else:
                decision['data_quality_score'] += 0.9
        
        # Recommend augmentations
        decision['augmentation_strategy'] = [
            'random_rotation',
            'horizontal_flip',
            'brightness_contrast',
            'elastic_transform' if len(perception['classes']) > 3 else 'affine'
        ]
        
        # Check for potential issues
        if perception['potential_issues']:
            decision['data_quality_score'] *= 0.8
            decision['recommended_preprocessing'].append('image_validation')
        
        return decision
    
    def act(self, decision: Dict) -> Dict:
        """Generate preprocessing recommendations"""
        action = {
            'preprocessing_pipeline': [],
            'augmentation_pipeline': [],
            'class_weights': None
        }
        
        # Build preprocessing pipeline
        if 'resize_to_uniform' in decision['recommended_preprocessing']:
            action['preprocessing_pipeline'].append({
                'type': 'resize',
                'params': {'size': (224, 224)}
            })
        
        # Build augmentation pipeline
        aug_map = {
            'random_rotation': {'type': 'rotate', 'params': {'limit': 30, 'p': 0.5}},
            'horizontal_flip': {'type': 'horizontal_flip', 'params': {'p': 0.5}},
            'brightness_contrast': {'type': 'random_brightness_contrast', 'params': {'p': 0.3}},
            'elastic_transform': {'type': 'elastic_transform', 'params': {'p': 0.2}}
        }
        
        for aug in decision['augmentation_strategy']:
            if aug in aug_map:
                action['augmentation_pipeline'].append(aug_map[aug])
        
        return action

class ModelArchitectAgent(BaseAgent):
    """Agent responsible for model selection and architecture design"""
    
    def __init__(self, config: AgentConfig):
        super().__init__(config)
        self.model_registry = {
            'efficientnet_b3': {
                'model': models.efficientnet_b3,
                'input_size': 300,
                'pretrained': True,
                'performance_tier': 'high',
                'inference_speed': 'fast'
            },
            'resnet50': {
                'model': models.resnet50,
                'input_size': 224,
                'pretrained': True,
                'performance_tier': 'medium',
                'inference_speed': 'fast'
            },
            'densenet121': {
                'model': models.densenet121,
                'input_size': 224,
                'pretrained': True,
                'performance_tier': 'medium',
                'inference_speed': 'medium'
            },
            'vit_b_16': {
                'model': models.vit_b_16,
                'input_size': 224,
                'pretrained': True,
                'performance_tier': 'high',
                'inference_speed': 'slow'
            },
            'swin_t': {
                'model': models.swin_t,
                'input_size': 224,
                'pretrained': True,
                'performance_tier': 'high',
                'inference_speed': 'medium'
            },
            'convnext_tiny': {
                'model': models.convnext_tiny,
                'input_size': 224,
                'pretrained': True,
                'performance_tier': 'high',
                'inference_speed': 'fast'
            }
        }
        
    def perceive(self, data_stats: Dict) -> Dict:
        """Analyze data characteristics for model selection"""
        perception = {
            'num_classes': len(data_stats['classes']),
            'dataset_size': sum(data_stats['image_counts'].values()),
            'class_distribution': data_stats['image_counts'],
            'image_complexity': 'medium',  # MRI images
            'task_type': 'classification'
        }
        
        # Estimate complexity based on class count
        if perception['num_classes'] <= 2:
            perception['task_complexity'] = 'binary'
        elif perception['num_classes'] <= 10:
            perception['task_complexity'] = 'multiclass_medium'
        else:
            perception['task_complexity'] = 'multiclass_large'
            
        return perception
    
    def reason(self, perception: Dict) -> Dict:
        """Select optimal model architecture"""
        decision = {
            'primary_model': None,
            'ensemble_models': [],
            'architecture_modifications': [],
            'training_strategy': {}
        }
        
        dataset_size = perception['dataset_size']
        num_classes = perception['num_classes']
        
        # Model selection logic
        if dataset_size > 5000:
            # Large dataset - can use more complex models
            decision['primary_model'] = 'efficientnet_b3'
            decision['ensemble_models'] = ['convnext_tiny', 'swin_t']
            decision['training_strategy']['epochs'] = 50
            decision['training_strategy']['batch_size'] = 32
        elif dataset_size > 2000:
            decision['primary_model'] = 'resnet50'
            decision['ensemble_models'] = ['densenet121']
            decision['training_strategy']['epochs'] = 40
            decision['training_strategy']['batch_size'] = 64
        else:
            decision['primary_model'] = 'densenet121'
            decision['training_strategy']['epochs'] = 30
            decision['training_strategy']['batch_size'] = 32
        
        # Architecture modifications
        decision['architecture_modifications'] = [
            'add_dropout',
            'add_batch_norm',
            'custom_classifier_head'
        ]
        
        return decision
    
    def act(self, decision: Dict) -> nn.Module:
        """Build and return the selected model"""
        primary_model_name = decision['primary_model']
        model_config = self.model_registry[primary_model_name]
        
        # Load pretrained model
        model = model_config['model'](weights='DEFAULT')
        
        # Get the number of input features for classifier
        if hasattr(model, 'classifier'):
            if isinstance(model.classifier, nn.Sequential):
                in_features = model.classifier[-1].in_features
            else:
                in_features = model.classifier.in_features
            # Replace classifier
            model.classifier = nn.Sequential(
                nn.Dropout(0.3),
                nn.Linear(in_features, 512),
                nn.ReLU(),
                nn.BatchNorm1d(512),
                nn.Dropout(0.3),
                nn.Linear(512, 256),
                nn.ReLU(),
                nn.BatchNorm1d(256),
                nn.Linear(256, 4)  # 4 classes
            )
        elif hasattr(model, 'fc'):
            in_features = model.fc.in_features
            model.fc = nn.Sequential(
                nn.Dropout(0.3),
                nn.Linear(in_features, 512),
                nn.ReLU(),
                nn.BatchNorm1d(512),
                nn.Dropout(0.3),
                nn.Linear(512, 4)
            )
        elif hasattr(model, 'heads'):
            in_features = model.heads.head.in_features
            model.heads.head = nn.Linear(in_features, 4)
        
        return model

class TrainingAgent(BaseAgent):
    """Agent responsible for model training and optimization"""
    
    def __init__(self, config: AgentConfig):
        super().__init__(config)
        self.best_model_state = None
        self.training_history = []
        self.early_stop_counter = 0
        
    def perceive(self, training_state: Dict) -> Dict:
        """Monitor training progress"""
        perception = {
            'current_loss': training_state.get('loss', float('inf')),
            'current_accuracy': training_state.get('accuracy', 0),
            'validation_loss': training_state.get('val_loss', float('inf')),
            'validation_accuracy': training_state.get('val_accuracy', 0),
            'learning_rate': training_state.get('lr', 0),
            'epoch': training_state.get('epoch', 0),
            'gradient_norm': training_state.get('grad_norm', 0),
            'time_elapsed': training_state.get('time', 0)
        }
        
        # Detect issues
        if perception['gradient_norm'] > 10:
            perception['issue'] = 'exploding_gradients'
        elif perception['gradient_norm'] < 1e-6:
            perception['issue'] = 'vanishing_gradients'
        elif perception['validation_loss'] > perception['training_loss'] * 1.5:
            perception['issue'] = 'overfitting'
        
        return perception
    
    def reason(self, perception: Dict) -> Dict:
        """Decide on training adjustments"""
        decision = {
            'continue_training': True,
            'adjust_learning_rate': False,
            'new_lr': perception['learning_rate'],
            'early_stop': False,
            'save_checkpoint': False,
            'reduce_overfitting': None
        }
        
        # Early stopping logic
        if len(self.performance_history) > 5:
            recent_val_losses = [h.get('val_loss', float('inf')) for h in self.performance_history[-5:]]
            if all(recent_val_losses[i] <= recent_val_losses[i+1] for i in range(len(recent_val_losses)-1)):
                self.early_stop_counter += 1
                if self.early_stop_counter >= 7:
                    decision['early_stop'] = True
            else:
                self.early_stop_counter = 0
        
        # Learning rate adjustment
        if perception.get('issue') == 'overfitting':
            decision['reduce_overfitting'] = 'increase_dropout'
            decision['adjust_learning_rate'] = True
            decision['new_lr'] = perception['learning_rate'] * 0.5
        
        # Checkpoint saving
        if perception['validation_accuracy'] > max([h.get('val_accuracy', 0) for h in self.performance_history] + [0]):
            decision['save_checkpoint'] = True
        
        return decision
    
    def act(self, decision: Dict) -> Dict:
        """Execute training decisions"""
        action = {
            'stop_training': decision['early_stop'],
            'new_learning_rate': decision['new_lr'] if decision['adjust_learning_rate'] else None,
            'save_model': decision['save_checkpoint'],
            'apply_regularization': decision.get('reduce_overfitting')
        }
        return action

class DiagnosticAgent(BaseAgent):
    """Agent for model interpretation and diagnosis"""
    
    def __init__(self, config: AgentConfig):
        super().__init__(config)
        self.cam_extractor = None
        self.attention_maps = []
        
    def perceive(self, model_output: Dict) -> Dict:
        """Analyze model predictions and behavior"""
        perception = {
            'predictions': model_output.get('predictions', []),
            'probabilities': model_output.get('probabilities', []),
            'true_labels': model_output.get('true_labels', []),
            'confidence_scores': model_output.get('confidence', []),
            'misclassifications': [],
            'uncertain_predictions': []
        }
        
        # Identify misclassifications
        for i, (pred, true, prob) in enumerate(zip(
            perception['predictions'], 
            perception['true_labels'],
            perception['probabilities']
        )):
            if pred != true:
                perception['misclassifications'].append({
                    'index': i,
                    'predicted': pred,
                    'true': true,
                    'confidence': max(prob)
                })
            elif max(prob) < self.config.confidence_threshold:
                perception['uncertain_predictions'].append({
                    'index': i,
                    'prediction': pred,
                    'confidence': max(prob)
                })
        
        return perception
    
    def reason(self, perception: Dict) -> Dict:
        """Diagnose issues and suggest improvements"""
        decision = {
            'diagnosis': [],
            'recommendations': [],
            'model_health': 'good'
        }
        
        # Analyze error patterns
        misclass_count = len(perception['misclassifications'])
        total_samples = len(perception['predictions'])
        error_rate = misclass_count / total_samples if total_samples > 0 else 0
        
        if error_rate > 0.2:
            decision['model_health'] = 'poor'
            decision['diagnosis'].append('high_error_rate')
            decision['recommendations'].append('collect_more_data')
            decision['recommendations'].append('try_different_architecture')
        elif error_rate > 0.1:
            decision['model_health'] = 'fair'
            decision['diagnosis'].append('moderate_error_rate')
            decision['recommendations'].append('fine_tune_hyperparameters')
        
        # Check confidence calibration
        if len(perception['uncertain_predictions']) / total_samples > 0.15:
            decision['diagnosis'].append('poor_calibration')
            decision['recommendations'].append('apply_temperature_scaling')
        
        return decision
    
    def act(self, decision: Dict) -> Dict:
        """Generate diagnostic report and actions"""
        action = {
            'diagnostic_report': {
                'model_health': decision['model_health'],
                'issues_found': decision['diagnosis'],
                'recommended_actions': decision['recommendations']
            },
            'require_retraining': decision['model_health'] == 'poor',
            'confidence_threshold_adjustment': 0.7 if decision['model_health'] == 'fair' else 0.85
        }
        return action

class MultiAgentSystem:
    """Coordinator for multiple AI agents"""
    
    def __init__(self):
        self.agents = {}
        self.communication_log = []
        self.task_queue = []
        
    def register_agent(self, agent: BaseAgent):
        """Register a new agent to the system"""
        self.agents[agent.config.name] = agent
        print(f"✅ Agent '{agent.config.name}' registered successfully")
        
    def broadcast_message(self, source: str, message: Dict):
        """Broadcast message to all agents"""
        for agent_name, agent in self.agents.items():
            if agent_name != source:
                response = agent.communicate(message)
                self.communication_log.append({
                    'from': source,
                    'to': agent_name,
                    'message': message,
                    'response': response,
                    'timestamp': datetime.datetime.now().isoformat()
                })
                
    def execute_pipeline(self, initial_data: Any) -> Dict:
        """Execute the full agent pipeline"""
        results = {}
        
        # Data Analysis Phase
        if 'DataAnalyzer' in self.agents:
            print("\n📊 Data Analysis Phase...")
            perception = self.agents['DataAnalyzer'].perceive(initial_data)
            decision = self.agents['DataAnalyzer'].reason(perception)
            action = self.agents['DataAnalyzer'].act(decision)
            results['data_analysis'] = {
                'perception': perception,
                'decision': decision,
                'action': action
            }
            
        # Model Architecture Phase
        if 'ModelArchitect' in self.agents and 'data_analysis' in results:
            print("\n🏗️ Model Architecture Phase...")
            perception = self.agents['ModelArchitect'].perceive(
                results['data_analysis']['perception']
            )
            decision = self.agents['ModelArchitect'].reason(perception)
            action = self.agents['ModelArchitect'].act(decision)
            results['model_architecture'] = {
                'perception': perception,
                'decision': decision,
                'action': action
            }
            
        return results


##  Dataset and DataLoader Implementation

In [4]:
class BrainTumorDataset(Dataset):
    """Advanced dataset class with intelligent preprocessing"""
    
    def __init__(self, root_dir: str, transform=None, phase: str = 'train',
                 use_advanced_augmentation: bool = True):
        self.root_dir = Path(root_dir)
        self.phase = phase
        self.transform = transform
        self.use_advanced_augmentation = use_advanced_augmentation
        
        # Load all image paths and labels
        self.images = []
        self.labels = []
        self.class_names = []
        
        # Get class names
        if (self.root_dir / 'Training').exists():
            self.class_names = sorted([d.name for d in (self.root_dir / 'Training').iterdir() if d.is_dir()])
        else:
            self.class_names = sorted([d.name for d in self.root_dir.iterdir() if d.is_dir()])
        
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.class_names)}
        self.idx_to_class = {idx: cls_name for cls_name, idx in self.class_to_idx.items()}
        
        # Load images based on phase
        if phase == 'train':
            data_dir = self.root_dir / 'Training'
        else:
            data_dir = self.root_dir / 'Testing'
        
        for class_name in self.class_names:
            class_dir = data_dir / class_name
            if class_dir.exists():
                for img_path in class_dir.glob('*'):
                    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                        self.images.append(str(img_path))
                        self.labels.append(self.class_to_idx[class_name])
        
        # Advanced augmentation for training
        self.advanced_augmentation = A.Compose([
            A.RandomResizedCrop(224, 224, scale=(0.8, 1.0)),
            A.Rotate(limit=30, p=0.5),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
            A.GaussNoise(var_limit=(10.0, 30.0), p=0.3),
            A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.2),
            A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.2),
            A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ]) if use_advanced_augmentation and phase == 'train' else None
        
        self.basic_transform = T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        print(f"Loaded {len(self.images)} images for {phase} phase")
        print(f"Classes: {self.class_names}")
        
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        
        # Load image
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Apply transformations
        if self.phase == 'train' and self.use_advanced_augmentation:
            augmented = self.advanced_augmentation(image=image)
            image = augmented['image']
        else:
            image = Image.fromarray(image)
            image = self.basic_transform(image)
        
        return {
            'image': image,
            'label': label,
            'path': img_path,
            'class_name': self.idx_to_class[label]
        }

##  Advanced Model Architectures

In [5]:
class MultiScaleAttention(nn.Module):
    """Multi-scale attention mechanism for better feature extraction"""
    
    def __init__(self, in_channels: int, reduction: int = 16):
        super().__init__()
        self.in_channels = in_channels
        
        # Channel attention
        self.channel_attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, in_channels // reduction, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // reduction, in_channels, 1),
            nn.Sigmoid()
        )
        
        # Spatial attention
        self.spatial_attention = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // reduction, 1, 3, padding=1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        # Channel attention
        ca = self.channel_attention(x)
        x = x * ca
        
        # Spatial attention
        sa = self.spatial_attention(x)
        x = x * sa
        
        return x

class EnsembleModel(nn.Module):
    """Ensemble of multiple models for improved accuracy"""
    
    def __init__(self, num_classes: int = 4):
        super().__init__()
        self.num_classes = num_classes
        
        # Initialize multiple backbone models
        self.efficientnet = models.efficientnet_b3(weights='DEFAULT')
        self.convnext = models.convnext_tiny(weights='DEFAULT')
        self.swin = models.swin_t(weights='DEFAULT')
        
        # Replace classifiers
        in_features_eff = self.efficientnet.classifier[1].in_features
        self.efficientnet.classifier = nn.Identity()
        
        in_features_conv = self.convnext.classifier[2].in_features
        self.convnext.classifier = nn.Identity()
        
        in_features_swin = self.swin.head.in_features
        self.swin.head = nn.Identity()
        
        # Attention fusion
        total_features = in_features_eff + in_features_conv + in_features_swin
        self.attention_fusion = nn.Sequential(
            nn.Linear(total_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, total_features),
            nn.Sigmoid()
        )
        
        # Final classifier
        self.classifier = nn.Sequential(
            nn.Linear(total_features, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        # Extract features from each model
        feat_eff = self.efficientnet(x)
        feat_conv = self.convnext(x)
        feat_swin = self.swin(x)
        
        # Concatenate features
        combined = torch.cat([feat_eff, feat_conv, feat_swin], dim=1)
        
        # Apply attention-based fusion
        attention_weights = self.attention_fusion(combined)
        fused_features = combined * attention_weights
        
        # Classification
        output = self.classifier(fused_features)
        
        return output

class BrainTumorClassifier(nn.Module):
    """Advanced classifier with multiple improvements"""
    
    def __init__(self, base_model: str = 'efficientnet_b3', num_classes: int = 4,
                 pretrained: bool = True, dropout_rate: float = 0.3):
        super().__init__()
        
        self.base_model_name = base_model
        self.num_classes = num_classes
        
        # Model registry
        model_dict = {
            'efficientnet_b3': (models.efficientnet_b3, 1536),
            'efficientnet_b5': (models.efficientnet_b5, 2048),
            'resnet50': (models.resnet50, 2048),
            'resnet101': (models.resnet101, 2048),
            'densenet121': (models.densenet121, 1024),
            'densenet169': (models.densenet169, 1664),
            'vit_b_16': (models.vit_b_16, 768),
            'swin_t': (models.swin_t, 768),
            'convnext_tiny': (models.convnext_tiny, 768),
            'convnext_base': (models.convnext_base, 1024)
        }
        
        if base_model in model_dict:
            model_fn, feature_dim = model_dict[base_model]
            if pretrained:
                self.backbone = model_fn(weights='DEFAULT')
            else:
                self.backbone = model_fn(weights=None)
        else:
            raise ValueError(f"Model {base_model} not supported")
        
        # Store feature dimension
        self.feature_dim = feature_dim
        
        # Remove original classifier
        if hasattr(self.backbone, 'classifier'):
            if isinstance(self.backbone.classifier, nn.Sequential):
                if base_model.startswith('efficientnet'):
                    self.backbone.classifier = nn.Identity()
                else:
                    self.backbone.classifier = nn.Identity()
            else:
                self.backbone.classifier = nn.Identity()
        elif hasattr(self.backbone, 'fc'):
            self.backbone.fc = nn.Identity()
        elif hasattr(self.backbone, 'heads'):
            self.backbone.heads = nn.Identity()
        elif hasattr(self.backbone, 'head'):
            self.backbone.head = nn.Identity()
        
        # Multi-scale attention
        self.attention = MultiScaleAttention(feature_dim) if 'vit' not in base_model else None
        
        # Advanced classifier head
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 1024),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(1024),
            nn.Dropout(dropout_rate),
            
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate),
            
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout_rate * 0.5),
            
            nn.Linear(256, num_classes)
        )
        
        # Auxiliary classifier for deep supervision
        self.aux_classifier = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x, return_features: bool = False):
        # Extract features
        features = self.backbone(x)
        
        # Apply attention if available
        if self.attention is not None and len(features.shape) == 4:
            features = self.attention(features)
            features = F.adaptive_avg_pool2d(features, 1).flatten(1)
        
        # Main classification
        output = self.classifier(features)
        
        # Auxiliary output
        aux_output = self.aux_classifier(features)
        
        if return_features:
            return output, aux_output, features
        return output, aux_output


## Training Pipeline with Advanced Features


In [6]:
class Trainer:
    """Advanced trainer with multiple optimizations"""
    
    def __init__(self, model: nn.Module, train_loader: DataLoader,
                 val_loader: DataLoader, test_loader: DataLoader = None,
                 device: str = 'cuda', config: Dict = None):
        
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.device = device
        
        # Default configuration
        self.config = {
            'epochs': 50,
            'learning_rate': 1e-4,
            'weight_decay': 1e-4,
            'label_smoothing': 0.1,
            'gradient_clip': 1.0,
            'early_stopping_patience': 10,
            'scheduler': 'cosine',
            'optimizer': 'adamw',
            'use_amp': True,
            'aux_loss_weight': 0.3,
            'mixup_alpha': 0.2,
            'cutmix_alpha': 0.2
        }
        
        if config:
            self.config.update(config)
        
        # Setup optimizer
        if self.config['optimizer'] == 'adamw':
            self.optimizer = AdamW(
                model.parameters(),
                lr=self.config['learning_rate'],
                weight_decay=self.config['weight_decay']
            )
        elif self.config['optimizer'] == 'adam':
            self.optimizer = Adam(
                model.parameters(),
                lr=self.config['learning_rate'],
                weight_decay=self.config['weight_decay']
            )
        else:
            self.optimizer = SGD(
                model.parameters(),
                lr=self.config['learning_rate'],
                momentum=0.9,
                weight_decay=self.config['weight_decay']
            )
        
        # Setup scheduler
        if self.config['scheduler'] == 'cosine':
            self.scheduler = CosineAnnealingWarmRestarts(
                self.optimizer,
                T_0=10,
                T_mult=2,
                eta_min=1e-6
            )
        elif self.config['scheduler'] == 'reduce':
            self.scheduler = ReduceLROnPlateau(
                self.optimizer,
                mode='max',
                factor=0.5,
                patience=5
            )
        elif self.config['scheduler'] == 'onecycle':
            self.scheduler = OneCycleLR(
                self.optimizer,
                max_lr=self.config['learning_rate'],
                epochs=self.config['epochs'],
                steps_per_epoch=len(train_loader)
            )
        
        # Loss functions
        self.criterion = nn.CrossEntropyLoss(
            label_smoothing=self.config['label_smoothing']
        )
        
        # Mixed precision training
        self.scaler = GradScaler() if self.config['use_amp'] else None
        
        # Metrics
        self.train_metrics = torchmetrics.MetricCollection({
            'acc': torchmetrics.Accuracy(task='multiclass', num_classes=4),
            'f1': torchmetrics.F1Score(task='multiclass', num_classes=4),
            'precision': torchmetrics.Precision(task='multiclass', num_classes=4),
            'recall': torchmetrics.Recall(task='multiclass', num_classes=4)
        }).to(device)
        
        self.val_metrics = self.train_metrics.clone().to(device)
        
        # Training history
        self.history = {
            'train_loss': [], 'train_acc': [], 'train_f1': [],
            'val_loss': [], 'val_acc': [], 'val_f1': [],
            'learning_rates': []
        }
        
        # Best model tracking
        self.best_val_acc = 0
        self.best_model_state = None
        self.early_stop_counter = 0
        
    def mixup_data(self, x, y, alpha=1.0):
        """MixUp augmentation"""
        if alpha > 0:
            lam = np.random.beta(alpha, alpha)
        else:
            lam = 1
            
        batch_size = x.size()[0]
        index = torch.randperm(batch_size).to(self.device)
        
        mixed_x = lam * x + (1 - lam) * x[index, :]
        y_a, y_b = y, y[index]
        
        return mixed_x, y_a, y_b, lam
    
    def cutmix_data(self, x, y, alpha=1.0):
        """CutMix augmentation"""
        if alpha > 0:
            lam = np.random.beta(alpha, alpha)
        else:
            lam = 1
            
        batch_size = x.size()[0]
        index = torch.randperm(batch_size).to(self.device)
        
        # Generate random box
        bbx1, bby1, bbx2, bby2 = self.rand_bbox(x.size(), lam)
        
        # Adjust lambda
        lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size()[-1] * x.size()[-2]))
        
        # Apply CutMix
        mixed_x = x.clone()
        mixed_x[:, :, bbx1:bbx2, bby1:bby2] = x[index, :, bbx1:bbx2, bby1:bby2]
        
        y_a, y_b = y, y[index]
        
        return mixed_x, y_a, y_b, lam
    
    def rand_bbox(self, size, lam):
        """Generate random bounding box for CutMix"""
        W = size[2]
        H = size[3]
        cut_rat = np.sqrt(1. - lam)
        cut_w = int(W * cut_rat)
        cut_h = int(H * cut_rat)
        
        cx = np.random.randint(W)
        cy = np.random.randint(H)
        
        bbx1 = np.clip(cx - cut_w // 2, 0, W)
        bby1 = np.clip(cy - cut_h // 2, 0, H)
        bbx2 = np.clip(cx + cut_w // 2, 0, W)
        bby2 = np.clip(cy + cut_h // 2, 0, H)
        
        return bbx1, bby1, bbx2, bby2
    
    def train_epoch(self):
        """Train for one epoch"""
        self.model.train()
        total_loss = 0
        self.train_metrics.reset()
        
        pbar = tqdm(self.train_loader, desc='Training')
        for batch_idx, batch in enumerate(pbar):
            images = batch['image'].to(self.device)
            labels = batch['label'].to(self.device)
            
            # Apply MixUp or CutMix
            use_mixup = random.random() < 0.5 if self.config['mixup_alpha'] > 0 else False
            
            if use_mixup:
                images, labels_a, labels_b, lam = self.mixup_data(
                    images, labels, self.config['mixup_alpha']
                )
            else:
                use_cutmix = random.random() < 0.3 if self.config['cutmix_alpha'] > 0 else False
                if use_cutmix:
                    images, labels_a, labels_b, lam = self.cutmix_data(
                        images, labels, self.config['cutmix_alpha']
                    )
            
            # Forward pass with mixed precision
            with autocast(enabled=self.config['use_amp']):
                outputs, aux_outputs = self.model(images)
                
                if use_mixup or use_cutmix:
                    loss = lam * self.criterion(outputs, labels_a) + \
                           (1 - lam) * self.criterion(outputs, labels_b)
                    aux_loss = lam * self.criterion(aux_outputs, labels_a) + \
                              (1 - lam) * self.criterion(aux_outputs, labels_b)
                else:
                    loss = self.criterion(outputs, labels)
                    aux_loss = self.criterion(aux_outputs, labels)
                
                total_loss = loss + self.config['aux_loss_weight'] * aux_loss
            
            # Backward pass
            self.optimizer.zero_grad()
            
            if self.config['use_amp']:
                self.scaler.scale(total_loss).backward()
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(),
                    self.config['gradient_clip']
                )
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(),
                    self.config['gradient_clip']
                )
                self.optimizer.step()
            
            # Update metrics
            if not (use_mixup or use_cutmix):
                self.train_metrics.update(outputs.softmax(dim=1), labels)
            
            # Update progress bar
            pbar.set_postfix({
                'loss': f"{total_loss.item():.4f}",
                'lr': f"{self.optimizer.param_groups[0]['lr']:.2e}"
            })
        
        # Compute epoch metrics
        metrics = self.train_metrics.compute()
        return total_loss.item() / len(self.train_loader), metrics
    
    def validate(self):
        """Validation step"""
        self.model.eval()
        total_loss = 0
        self.val_metrics.reset()
        
        all_preds = []
        all_labels = []
        all_probs = []
        
        with torch.no_grad():
            pbar = tqdm(self.val_loader, desc='Validation')
            for batch in pbar:
                images = batch['image'].to(self.device)
                labels = batch['label'].to(self.device)
                
                outputs, _ = self.model(images)
                loss = self.criterion(outputs, labels)
                total_loss += loss.item()
                
                probs = outputs.softmax(dim=1)
                self.val_metrics.update(probs, labels)
                
                all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())
        
        metrics = self.val_metrics.compute()
        return total_loss / len(self.val_loader), metrics, all_preds, all_labels, all_probs
    
    def train(self):
        """Full training loop"""
        print("\n" + "="*80)
        print("STARTING TRAINING")
        print("="*80)
        
        for epoch in range(self.config['epochs']):
            print(f"\nEpoch {epoch+1}/{self.config['epochs']}")
            print("-" * 40)
            
            # Training
            train_loss, train_metrics = self.train_epoch()
            
            # Validation
            val_loss, val_metrics, _, _, _ = self.validate()
            
            # Update scheduler
            if isinstance(self.scheduler, ReduceLROnPlateau):
                self.scheduler.step(val_metrics['acc'])
            elif isinstance(self.scheduler, CosineAnnealingWarmRestarts):
                self.scheduler.step()
            elif isinstance(self.scheduler, OneCycleLR):
                self.scheduler.step()
            
            # Save history
            self.history['train_loss'].append(train_loss)
            self.history['train_acc'].append(train_metrics['acc'].item())
            self.history['train_f1'].append(train_metrics['f1'].item())
            self.history['val_loss'].append(val_loss)
            self.history['val_acc'].append(val_metrics['acc'].item())
            self.history['val_f1'].append(val_metrics['f1'].item())
            self.history['learning_rates'].append(
                self.optimizer.param_groups[0]['lr']
            )
            
            # Print metrics
            print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_metrics['acc']:.4f} | Train F1: {train_metrics['f1']:.4f}")
            print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_metrics['acc']:.4f} | Val F1: {val_metrics['f1']:.4f}")
            
            # Save best model
            if val_metrics['acc'] > self.best_val_acc:
                self.best_val_acc = val_metrics['acc']
                self.best_model_state = self.model.state_dict().copy()
                self.early_stop_counter = 0
                print(f"✅ New best model! Val Acc: {self.best_val_acc:.4f}")
            else:
                self.early_stop_counter += 1
            
            # Early stopping
            if self.early_stop_counter >= self.config['early_stopping_patience']:
                print(f"\n⚠️ Early stopping triggered after {epoch+1} epochs")
                break
        
        # Load best model
        self.model.load_state_dict(self.best_model_state)
        print(f"\n🏆 Training completed! Best Val Acc: {self.best_val_acc:.4f}")
        
        return self.history

##  Model Evaluation and Interpretation

In [7]:

class ModelEvaluator:
    """Comprehensive model evaluation and interpretation"""
    
    def __init__(self, model: nn.Module, device: str = 'cuda'):
        self.model = model.to(device)
        self.device = device
        self.model.eval()
        
    def evaluate(self, test_loader: DataLoader) -> Dict:
        """Full model evaluation"""
        all_preds = []
        all_labels = []
        all_probs = []
        
        with torch.no_grad():
            for batch in tqdm(test_loader, desc='Evaluating'):
                images = batch['image'].to(self.device)
                labels = batch['label']
                
                outputs, _ = self.model(images)
                probs = outputs.softmax(dim=1).cpu().numpy()
                preds = outputs.argmax(dim=1).cpu().numpy()
                
                all_preds.extend(preds)
                all_labels.extend(labels.numpy())
                all_probs.extend(probs)
        
        # Calculate metrics
        accuracy = accuracy_score(all_labels, all_preds)
        precision, recall, f1, _ = precision_recall_fscore_support(
            all_labels, all_preds, average='weighted'
        )
        
        # Confusion matrix
        cm = confusion_matrix(all_labels, all_preds)
        
        # Classification report
        report = classification_report(all_labels, all_preds, output_dict=True)
        
        # ROC-AUC for multiclass
        labels_bin = label_binarize(all_labels, classes=[0, 1, 2, 3])
        roc_auc = roc_auc_score(labels_bin, all_probs, average='macro', multi_class='ovr')
        
        # Cohen's Kappa
        kappa = cohen_kappa_score(all_labels, all_preds)
        
        # Matthews Correlation Coefficient
        mcc = matthews_corrcoef(all_labels, all_preds)
        
        results = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'roc_auc': roc_auc,
            'cohen_kappa': kappa,
            'matthews_corrcoef': mcc,
            'confusion_matrix': cm,
            'classification_report': report,
            'predictions': all_preds,
            'true_labels': all_labels,
            'probabilities': all_probs
        }
        
        return results
    
    def generate_gradcam(self, image_tensor: torch.Tensor, target_layer: str = None):
        """Generate GradCAM visualizations"""
        
        # Find target layer
        if target_layer is None:
            if hasattr(self.model, 'backbone'):
                if hasattr(self.model.backbone, 'features'):
                    target_layers = [self.model.backbone.features[-1]]
                elif hasattr(self.model.backbone, 'layer4'):
                    target_layers = [self.model.backbone.layer4[-1]]
                else:
                    # Default to last convolutional layer
                    for name, module in self.model.backbone.named_modules():
                        if isinstance(module, nn.Conv2d):
                            last_conv = module
                    target_layers = [last_conv]
            else:
                target_layers = [self.model.backbone.layer4[-1]]
        
        # Initialize GradCAM
        cam = GradCAM(model=self.model, target_layers=target_layers)
        
        # Generate CAM
        grayscale_cam = cam(input_tensor=image_tensor)
        
        return grayscale_cam


##  Visualization Functions

In [8]:
class Visualizer:
    """Advanced visualization tools"""
    
    @staticmethod
    def plot_training_history(history: Dict):
        """Plot training history"""
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=('Loss Curves', 'Accuracy Curves', 
                          'F1 Score Curves', 'Learning Rate'),
            vertical_spacing=0.12
        )
        
        # Loss curves
        fig.add_trace(
            go.Scatter(y=history['train_loss'], name='Train Loss',
                      line=dict(color='blue', width=2)),
            row=1, col=1
        )
        fig.add_trace(
            go.Scatter(y=history['val_loss'], name='Val Loss',
                      line=dict(color='red', width=2)),
            row=1, col=1
        )
        
        # Accuracy curves
        fig.add_trace(
            go.Scatter(y=history['train_acc'], name='Train Acc',
                      line=dict(color='blue', width=2)),
            row=1, col=2
        )
        fig.add_trace(
            go.Scatter(y=history['val_acc'], name='Val Acc',
                      line=dict(color='red', width=2)),
            row=1, col=2
        )
        
        # F1 curves
        fig.add_trace(
            go.Scatter(y=history['train_f1'], name='Train F1',
                      line=dict(color='blue', width=2)),
            row=2, col=1
        )
        fig.add_trace(
            go.Scatter(y=history['val_f1'], name='Val F1',
                      line=dict(color='red', width=2)),
            row=2, col=1
        )
        
        # Learning rate
        fig.add_trace(
            go.Scatter(y=history['learning_rates'], name='Learning Rate',
                      line=dict(color='green', width=2)),
            row=2, col=2
        )
        
        fig.update_layout(height=800, showlegend=True,
                         title_text="Training History")
        fig.show()
    
    @staticmethod
    def plot_confusion_matrix(cm: np.ndarray, class_names: List[str]):
        """Plot confusion matrix"""
        fig = px.imshow(
            cm,
            text_auto=True,
            labels=dict(x="Predicted", y="True", color="Count"),
            x=class_names,
            y=class_names,
            color_continuous_scale='Blues',
            title='Confusion Matrix'
        )
        
        fig.update_layout(
            width=600,
            height=600,
            title_x=0.5
        )
        fig.show()
        
        # Normalized confusion matrix
        cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        
        fig_norm = px.imshow(
            cm_norm,
            text_auto='.2%',
            labels=dict(x="Predicted", y="True", color="Percentage"),
            x=class_names,
            y=class_names,
            color_continuous_scale='Greens',
            title='Normalized Confusion Matrix'
        )
        
        fig_norm.update_layout(
            width=600,
            height=600,
            title_x=0.5
        )
        fig_norm.show()
    
    @staticmethod
    def plot_metrics_comparison(metrics: Dict):
        """Plot metrics comparison"""
        metrics_names = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']
        values = [metrics.get(m, 0) for m in metrics_names]
        
        fig = go.Figure(data=[
            go.Bar(
                x=metrics_names,
                y=values,
                text=[f'{v:.3f}' for v in values],
                textposition='auto',
                marker_color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
            )
        ])
        
        fig.update_layout(
            title='Model Performance Metrics',
            yaxis=dict(range=[0, 1], title='Score'),
            width=800,
            height=500,
            title_x=0.5
        )
        
        fig.show()


##  Main Execution Pipeline

In [9]:
def main():
    """Main execution pipeline"""
    
    print("\n" + "="*80)
    print("🤖 BRAIN TUMOR MRI CLASSIFICATION WITH AGENTIC AI")
    print("="*80)
    
    # Initialize Multi-Agent System
    print("\n📌 Initializing Agentic AI System...")
    mas = MultiAgentSystem()
    
    # Create and register agents
    data_agent = DataAnalysisAgent(
        AgentConfig(
            name='DataAnalyzer',
            role='Data Analysis Specialist',
            capabilities=['data_exploration', 'quality_assessment', 'preprocessing_recommendation']
        )
    )
    
    model_agent = ModelArchitectAgent(
        AgentConfig(
            name='ModelArchitect',
            role='Model Architecture Designer',
            capabilities=['model_selection', 'architecture_optimization', 'ensemble_design']
        )
    )
    
    training_agent = TrainingAgent(
        AgentConfig(
            name='TrainingAgent',
            role='Training Optimization Specialist',
            capabilities=['hyperparameter_tuning', 'training_monitoring', 'early_stopping']
        )
    )
    
    diagnostic_agent = DiagnosticAgent(
        AgentConfig(
            name='DiagnosticAgent',
            role='Model Diagnosis Expert',
            capabilities=['error_analysis', 'model_interpretation', 'performance_diagnosis']
        )
    )
    
    mas.register_agent(data_agent)
    mas.register_agent(model_agent)
    mas.register_agent(training_agent)
    mas.register_agent(diagnostic_agent)
    
    # Data path
    data_path = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset'
    
    # Execute agent pipeline
    print("\n📊 Running Agent Analysis Pipeline...")
    agent_results = mas.execute_pipeline(data_path)
    
    # Display agent insights
    print("\n" + "="*80)
    print("📋 AGENT INSIGHTS AND RECOMMENDATIONS")
    print("="*80)
    
    data_insights = agent_results['data_analysis']['decision']
    print(f"\n🔍 Data Quality Score: {data_insights['data_quality_score']:.2f}")
    print(f"📊 Class Balance: {data_insights['class_balance_status']}")
    print(f"🛠️ Recommended Preprocessing: {', '.join(data_insights['recommended_preprocessing'])}")
    print(f"🎨 Augmentation Strategy: {', '.join(data_insights['augmentation_strategy'])}")
    
    model_insights = agent_results['model_architecture']['decision']
    print(f"\n🏗️ Primary Model: {model_insights['primary_model']}")
    print(f"🔧 Ensemble Models: {', '.join(model_insights['ensemble_models'])}")
    print(f"⚙️ Training Strategy: {model_insights['training_strategy']}")
    
    # Create datasets
    print("\n" + "="*80)
    print("📦 CREATING DATASETS AND DATALOADERS")
    print("="*80)
    
    train_dataset = BrainTumorDataset(
        root_dir=data_path,
        phase='Training',
        use_advanced_augmentation=True
    )
    
    val_dataset = BrainTumorDataset(
        root_dir=data_path,
        phase='Testing',
        use_advanced_augmentation=False
    )
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=model_insights['training_strategy']['batch_size'],
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=model_insights['training_strategy']['batch_size'],
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )
    
    # Create model
    print("\n" + "="*80)
    print("🏗️ BUILDING MODEL")
    print("="*80)
    
    model = BrainTumorClassifier(
        base_model=model_insights['primary_model'],
        num_classes=4,
        pretrained=True,
        dropout_rate=0.3
    )
    
    print(f"\n✅ Model created: {model_insights['primary_model']}")
    print(f"📊 Total parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"🔄 Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    
    # Training configuration
    training_config = {
        'epochs': model_insights['training_strategy']['epochs'],
        'learning_rate': 1e-4,
        'batch_size': model_insights['training_strategy']['batch_size'],
        'weight_decay': 1e-4,
        'label_smoothing': 0.1,
        'gradient_clip': 1.0,
        'early_stopping_patience': 10,
        'scheduler': 'cosine',
        'optimizer': 'adamw',
        'use_amp': True,
        'aux_loss_weight': 0.3,
        'mixup_alpha': 0.2,
        'cutmix_alpha': 0.2
    }
    
    # Initialize trainer
    print("\n" + "="*80)
    print("🚀 STARTING TRAINING")
    print("="*80)
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    trainer = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        config=training_config
    )
    
    # Train model
    history = trainer.train()
    
    # Evaluate model
    print("\n" + "="*80)
    print("📊 MODEL EVALUATION")
    print("="*80)
    
    evaluator = ModelEvaluator(model, device)
    evaluation_results = evaluator.evaluate(val_loader)
    
    # Display results
    print(f"\n🎯 Test Accuracy: {evaluation_results['accuracy']:.4f}")
    print(f"📈 Test F1 Score: {evaluation_results['f1_score']:.4f}")
    print(f"🎪 ROC-AUC Score: {evaluation_results['roc_auc']:.4f}")
    print(f"📊 Cohen's Kappa: {evaluation_results['cohen_kappa']:.4f}")
    print(f"🔗 Matthews Correlation: {evaluation_results['matthews_corrcoef']:.4f}")
    
    # Visualizations
    print("\n" + "="*80)
    print("📈 GENERATING VISUALIZATIONS")
    print("="*80)
    
    visualizer = Visualizer()
    
    # Plot training history
    print("\n📊 Plotting training history...")
    visualizer.plot_training_history(history)
    
    # Plot confusion matrix
    print("\n🎯 Plotting confusion matrix...")
    class_names = train_dataset.class_names
    visualizer.plot_confusion_matrix(
        evaluation_results['confusion_matrix'],
        class_names
    )
    
    # Plot metrics comparison
    print("\n📈 Plotting metrics comparison...")
    visualizer.plot_metrics_comparison(evaluation_results)
    
    # Save model
    print("\n" + "="*80)
    print("💾 SAVING MODEL")
    print("="*80)
    
    model_save_path = 'brain_tumor_classifier_best.pth'
    torch.save({
        'model_state_dict': model.state_dict(),
        'class_names': class_names,
        'model_config': {
            'base_model': model_insights['primary_model'],
            'num_classes': 4
        },
        'evaluation_metrics': evaluation_results
    }, model_save_path)
    
    print(f"✅ Model saved to {model_save_path}")
    
    # Generate final report
    print("\n" + "="*80)
    print("📋 FINAL REPORT")
    print("="*80)
    
    print(f"""
    🏆 TRAINING COMPLETED SUCCESSFULLY
    
    Model Architecture: {model_insights['primary_model']}
    Best Validation Accuracy: {trainer.best_val_acc:.4f}
    Test Accuracy: {evaluation_results['accuracy']:.4f}
    Test F1 Score: {evaluation_results['f1_score']:.4f}
    
    Training Duration: {len(history['train_loss'])} epochs
    Early Stopping: {'Yes' if trainer.early_stop_counter >= training_config['early_stopping_patience'] else 'No'}
    
    Classification Report:
    {pd.DataFrame(evaluation_results['classification_report']).transpose().to_string()}
    """)
    
    print("\n" + "="*80)
    print("✅ PIPELINE EXECUTION COMPLETED SUCCESSFULLY!")
    print("="*80)
    
    return model, history, evaluation_results


if __name__ == "__main__":
    model, history, evaluation_results = main()



🤖 BRAIN TUMOR MRI CLASSIFICATION WITH AGENTIC AI

📌 Initializing Agentic AI System...
✅ Agent 'DataAnalyzer' registered successfully
✅ Agent 'ModelArchitect' registered successfully
✅ Agent 'TrainingAgent' registered successfully
✅ Agent 'DiagnosticAgent' registered successfully

📊 Running Agent Analysis Pipeline...

📊 Data Analysis Phase...

🏗️ Model Architecture Phase...
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 193MB/s]



📋 AGENT INSIGHTS AND RECOMMENDATIONS

🔍 Data Quality Score: 0.70
📊 Class Balance: balanced
🛠️ Recommended Preprocessing: resize_to_uniform
🎨 Augmentation Strategy: random_rotation, horizontal_flip, brightness_contrast, elastic_transform

🏗️ Primary Model: efficientnet_b3
🔧 Ensemble Models: convnext_tiny, swin_t
⚙️ Training Strategy: {'epochs': 50, 'batch_size': 32}

📦 CREATING DATASETS AND DATALOADERS
Loaded 1600 images for Training phase
Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Loaded 1600 images for Testing phase
Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']

🏗️ BUILDING MODEL

✅ Model created: efficientnet_b3
📊 Total parameters: 13,770,321
🔄 Trainable parameters: 13,770,321

🚀 STARTING TRAINING

STARTING TRAINING

Epoch 1/50
----------------------------------------


Training:   0%|          | 0/50 [00:00<?, ?it/s]

Validation:   0%|          | 0/50 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]


🎯 Test Accuracy: 1.0000
📈 Test F1 Score: 1.0000
🎪 ROC-AUC Score: 1.0000
📊 Cohen's Kappa: 1.0000
🔗 Matthews Correlation: 1.0000

📈 GENERATING VISUALIZATIONS

📊 Plotting training history...



🎯 Plotting confusion matrix...



📈 Plotting metrics comparison...



💾 SAVING MODEL
✅ Model saved to brain_tumor_classifier_best.pth

📋 FINAL REPORT

    🏆 TRAINING COMPLETED SUCCESSFULLY
    
    Model Architecture: efficientnet_b3
    Best Validation Accuracy: 1.0000
    Test Accuracy: 1.0000
    Test F1 Score: 1.0000
    
    Training Duration: 28 epochs
    Early Stopping: Yes
    
    Classification Report:
                  precision  recall  f1-score  support
0                   1.0     1.0       1.0    400.0
1                   1.0     1.0       1.0    400.0
2                   1.0     1.0       1.0    400.0
3                   1.0     1.0       1.0    400.0
accuracy            1.0     1.0       1.0      1.0
macro avg           1.0     1.0       1.0   1600.0
weighted avg        1.0     1.0       1.0   1600.0
    

✅ PIPELINE EXECUTION COMPLETED SUCCESSFULLY!


##  Advanced Inference with GradCAM Visualization



In [10]:
def predict_single_image(model, image_path, class_names, device='cuda'):
    """Predict and visualize single image"""
    
    # Load and preprocess image
    transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    # Predict
    model.eval()
    with torch.no_grad():
        outputs, _ = model(image_tensor)
        probs = outputs.softmax(dim=1)
        pred_class = outputs.argmax(dim=1).item()
        confidence = probs[0][pred_class].item()
    
    # Generate GradCAM
    evaluator = ModelEvaluator(model, device)
    cam = evaluator.generate_gradcam(image_tensor)
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Original image
    axes[0].imshow(image)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    # GradCAM
    cam_image = show_cam_on_image(
        np.array(image.resize((224, 224))) / 255.0,
        cam[0],
        use_rgb=True
    )
    axes[1].imshow(cam_image)
    axes[1].set_title('GradCAM Visualization')
    axes[1].axis('off')
    
    # Class probabilities
    probs_np = probs[0].cpu().numpy()
    axes[2].bar(class_names, probs_np)
    axes[2].set_title(f'Prediction: {class_names[pred_class]}\nConfidence: {confidence:.2%}')
    axes[2].set_xticklabels(class_names, rotation=45)
    axes[2].set_ylim([0, 1])
    
    plt.tight_layout()
    plt.show()
    
    return {
        'predicted_class': class_names[pred_class],
        'confidence': confidence,
        'all_probabilities': {cls: prob for cls, prob in zip(class_names, probs_np)}
    }

# Example usage:
# result = predict_single_image(model, '/path/to/image.jpg', class_names)


##  Hyperparameter Optimization with Optuna


In [11]:
def objective(trial):
    """Optuna objective function for hyperparameter optimization"""
    
    # Suggest hyperparameters
    lr = trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    optimizer_name = trial.suggest_categorical('optimizer', ['adam', 'adamw', 'sgd'])
    
    # Create dataloaders with suggested batch size
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2
    )
    
    # Create model
    model = BrainTumorClassifier(
        base_model='efficientnet_b3',
        num_classes=4,
        dropout_rate=dropout_rate
    )
    
    # Training config
    config = {
        'epochs': 10,  # Reduced for optimization
        'learning_rate': lr,
        'weight_decay': weight_decay,
        'optimizer': optimizer_name,
        'batch_size': batch_size,
        'early_stopping_patience': 5
    }
    
    # Train
    trainer = Trainer(model, train_loader, val_loader, config=config)
    history = trainer.train()
    
    # Return best validation accuracy
    return trainer.best_val_acc

# Uncomment to run hyperparameter optimization
# study = optuna.create_study(direction='maximize')
# study.optimize(objective, n_trials=20)
# print(f"Best parameters: {study.best_params}")
# print(f"Best accuracy: {study.best_value}")

print("\n✅ Full implementation ready for Kaggle notebook execution!")
print("📝 This comprehensive agentic AI system includes:")
print("  • Multi-agent architecture for autonomous decision making")
print("  • Advanced data analysis and preprocessing recommendations")
print("  • Intelligent model selection and architecture design")
print("  • State-of-the-art training with MixUp, CutMix, and mixed precision")
print("  • Comprehensive evaluation and visualization suite")
print("  • GradCAM for model interpretability")
print("  • Hyperparameter optimization capabilities")


✅ Full implementation ready for Kaggle notebook execution!
📝 This comprehensive agentic AI system includes:
  • Multi-agent architecture for autonomous decision making
  • Advanced data analysis and preprocessing recommendations
  • Intelligent model selection and architecture design
  • State-of-the-art training with MixUp, CutMix, and mixed precision
  • Comprehensive evaluation and visualization suite
  • GradCAM for model interpretability
  • Hyperparameter optimization capabilities
